Backpropagation practice before moving into RNNs, Wavenet CNN, and transformers.

In [6]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [7]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [8]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [9]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
    X, Y = [], []
    
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # crop and append
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])    # 80%
Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte,  Yte  = build_dataset(words[n2:])    # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [10]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [11]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),                             generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializing many of these parameters in non-standard ways
# because sometimes initializing with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True


4137


In [12]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch 
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

Make sure you understand the step by step processes of linear layers (relatively simple), but more importantly, batchnorms and cross-entropy. Their trickier.

In [13]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# Cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
          embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.3379, grad_fn=<NegBackward0>)

In [16]:
logprobs.shape

torch.Size([32, 27])

In [44]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one


# loss = -(a + b + c) / 3 = -1/3a + -1/3b + -1/3c
# dloss/da = -1/3, or more generic: -1/n
# loss is defined at the negation of the max value in each row of logprobs averaged. In this case, the mean of 32 numbers
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
# Check:
cmp('logprobs', dlogprobs, logprobs)


# logprobs = probs.log() --> y = log(x)
# dlogprobs/dprobs = 1/probs
# dprobs = 1/probs --> however, it must be chained with the upstream gradient dlogprobs:
dprobs = (1/probs) * dlogprobs
cmp('probs', dprobs, probs)


# probs = counts * counts_sum_inv --> [32, 27] * [32, 1]
# If probs = counts * counts_sum_inv, dcounts_sum_inv = dprobs (also must be chained upstream:)

# let c = a * b, a = [3, 3], b = [3, 1] --> b gets replicated in 2 additional columns to match the shape of a
# See how b1 is multiplied in the entire first row, b2 in the second row, and b3 in the third row:
# c = [[a11*b1, a12*b1, a13*b1],
#      [a21*b2, a22*b2, a23*b2],
#      [a31*b3, a32*b3, a33*b3]]

# Looking back at probs = counts * counts_sum_inv, it looks like 1 operation, but in reality theres 2
# First, counts_sum_inv ir replicated in each column to match the shape of counts
# Second, they are multiplied
# In broadcasting, counts_sum_inv[1, 0] is replicated in every column, so the gradient is the sum of the gradients in each column (sum(1))
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts = counts_sum_inv * dprobs
# cmp('counts', dcounts, counts) WILL NOT WORK YET, as counts contributes to counts_sum_inv, and directly into probs
# The gradient of counts is not fully determined yet


# counts_sum_inv = counts_sum**-1 --> y = x**-1
dcounts_sum = -(counts_sum**-2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)


# counts_sum = counts.sum(1, keepdim=True)
# a11, a12, a13 --> b1 = (a11 + a12 + a13)
# a21, a22, a23 --> b2 = (a21 + a22 + a23)
# a31, a32, a33 --> b3 = (a31 + a32 + a33)
# recall that for addition operations, the gradient just flows through, equally to each child
# b1 flows equally to a11, a12, a13, and so on for b2 and b3
# so dcounts would just be an array of 1s in the same shape as counts chained with the upstream derivative
# Use plus equal since we must sum this with the perviously calculated count gradient from the direct path to probs
dcounts += torch.ones_like(counts) * dcounts_sum
cmp('counts', dcounts, counts)


# counts = norm_logits.exp() --> y = exp(x), dy/dx = y = exp(x)
dnorm_logits = counts * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)


# norm_logits = logits - logit_maxes --> y = a - b --> [32, 27] - [32, 1] (broadcasting)
# dlogit_maxes = -(torch.ones_like(norm_logits)).sum(1, keepdim=True) * dnorm_logits
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
# ^^^^^^^^^^^^^^^^^^^^^
# DO NOT UNDERSTAND WHY


# logit_maxes = logits.max(1, keepdim=True).values
dlogits = dnorm_logits.clone()

# norm_logits = logits - logit_maxes # subtract max for numerical stability
# dlogits = 1 * dlogit_maxes
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
cmp('logits', dlogits, logits)


# logits = h @ W2 + b2 --> y = a @ b + c, W2 is [64, 27], b2 is [27,] (therefore broadcasting is present, and the bias becomes a row vector)
# For something like this, its def best to write out the shapes on paper and see how each element of each input affects the output h:
# For eg. d = a @ b + c
# d11, d12   =====   [a11, a12] [b11, b12]   +   [c1, c2]
# d21, d22   =====   [a21, a22] [b21, b22]       [c1, c2]

#  d11 = [a11*b11 + a12*b21 + c1]
#  d12 = [a11*b12 + a12*b22 + c2]
#  d21 = [a21*b11 + a22*b21 + c1]
#  d22 = [a21*b12 + a22*b22 + c2]

# Can't rlly do the derivative calculations on here, but if you (reader) would like to see, check Karpathy's Makemore p4 video @ 46:15
# However, in general there are rules that can be applied to get the derivative easier:
# For the bias, the gradient just flows through, but since its broadcasted, we must sum the gradients for each column together (sum(0))
    # dh must be the same shape as h --> [32, 64]
    # dh must also be some kind of matric multiplication between dlogits (32, 27) and W2 (64, 27)
    # The only way to achieve this is to multiple dlogits by the transpose of W2:
dh = dlogits @ W2.T
cmp('h', dh, h)
# Same process can be applied to W2
dW2 = h.T @ dlogits
cmp('W2', dW2, W2)
# For b2, the only way to attain a shape of [27,] is to sum dlogits along the 0th axis, squashing each column to a single row
db2 = dlogits.sum(0)
cmp('b2', db2, b2)



# h = torch.tanh(hpreact) --> y = tanh(x), dy/dx = 1 - y**2
dhpreact = (1 - h**2) * dh
cmp('hpreact', dhpreact, hpreact)


# hpreact = bngain * bnraw + bnbias --> y = a * b + c, where a and b are elementwise multiplied (not cross product)
# shapes: [32, 64], [1, 64], [32, 64], and [1, 64] respectively
dbngain = (bnraw * dhpreact).sum(0, keepdim=True) # Since bngain is a ROW being broadcasted
dbnraw = (bngain * dhpreact) # No summing neeeded sincebnraw is not broadcasted in the operation
dbnbias = dhpreact.sum(0, keepdim=True) # Since bnbias is a ROW being broadcasted
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)


# bnraw = bndiff * bnvar_inv --> y = a * b
# Shapes: [32, 64], [32, 64], [1, 64]
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)


# bnvar_inv = (bnvar + 1e-5)**-0.5 --> y = (x + c)**-0.5
dbnvar = -0.5 * (bnvar + 1e-5)**-1.5 * dbnvar_inv
cmp('bnvar', dbnvar, bnvar)


# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# --> y = 1/(n-1) * sum(x)
# Shapes: [1, 64], [32, 64]
dbndiff2 = (1/(n-1)) * torch.ones_like(bndiff2) * dbnvar
cmp('bndiff2', dbndiff2, bndiff2)


# bnraw = bndiff * bnvar_inv
# bndiff2 = bndiff**2
dbndiff = (bnvar_inv * dbnraw) + (2 * bndiff * dbndiff2)
cmp('bndiff', dbndiff, bndiff)


# bndiff = hprebn - bnmeani
# Shapes: [32, 64], [32, 64], [1, 64]
dbnmeani = (-torch.ones_like(bndiff) * dbndiff).sum(0, keepdim=True)
cmp('bnmeani', dbnmeani, bnmeani)
dhprebn = dbndiff.clone() # Just cloning it because we don't want to mess with it's gradients with other changes to other variables

# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
dhprebn += 1/n * (torch.ones_like(hprebn) * dbnmeani)
cmp('hprebn', dhprebn, hprebn)


# hprebn = embcat @ W1 + b1 --> y = a * b + c, where a and b are cross producted, and c is broadcasted
# shapes: [32, 64], [32, 30], [30, 64], [64]
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn # dembcat and dW1 are derived by looking at the shapes to see what matches
db1 = dhprebn.sum(0) # pretty simple since its just an addition operation with broadcasting
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)


# embcat = emb.view(emb.shape[0], -1) --> y = x.view(a, b)
# shapes: [32, 30], [32, 3, 10]
demb = dembcat.view(emb.shape)
cmp('emb', demb, emb)


# emb = C[Xb] --> y = a[b], where a is the embedding matrix, and b is the indices
# shapes: [32, 3, 10], [27, 10], [32, 3]
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]
cmp('C', dC, C)


logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [ ]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())


In [46]:
# backward pass

# dL/dlogits - softmax(logits) - one_hot(Yb), where one_hot(Yb) = [0, 0, ...1 at correct class, 0, 0...]
# dLogits measures the gradient at of each logit
dlogits = F.softmax(logits, 1) # Softmax each row
dlogits[range(n), Yb] -= 1 # Subtract 1 at the correct class index. This causes the gradient to be negative no matter what.
# This causes the the logit of the correct class to be increased, and others (incorrect) to be decreased, ultimately improving the probability for next time.
dlogits /= n # Average over the batch to avoid logit values from growing with batch size

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits          | exact: False | approximate: True  | maxdiff: 4.423782229423523e-09


In [ ]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),                             generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written (TODO)
#with torch.no_grad():

# kick off optimization
for i in range(max_steps):
    
    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

    # forward pass
    emb = C[Xb] # embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
    # Linear layer 1
    hprebn = embcat @ W1 + b1 # hidden layer pre-activation
    # BatchNorm layer
    # -------------------------------------------------------------------------
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    # -------------------------------------------------------------------------
    # Non-linearity
    h = torch.tanh(hpreact) # hidden layer
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward() # use this for correctness comparisons, delete it later!

    # manual backprop! #swole_doge_meme
    # -------------------------------------------------------------------------
    # YOUR CODE HERE :)
    dC, dW1, db1, dW2, db2, dbngain, dbnbias = None, None, None, None, None, None, None
    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    # -------------------------------------------------------------------------

    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p, grad in zip(parameters, grads):
        p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
        #p.data += -lr * grad # new way of swole doge TODO: enable

    # track stats
    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    if i >= 100: # TODO: delete early breaking when you're ready to train the full net
        break

In [ ]:
# useful for checking your gradients
# for p,g in zip(parameters, grads):
#     cmp(str(tuple(p.shape)), g, p)

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
    # pass the training set through
    emb = C[Xtr]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    # measure the mean/std over the entire training set
    bnmean = hpreact.mean(0, keepdim=True)
    bnvar = hpreact.var(0, keepdim=True, unbiased=True)


In [ ]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
    x,y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x] # (N, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
    h = torch.tanh(hpreact) # (N, n_hidden)
    logits = h @ W2 + b2 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# I achieved:
# train 2.0718822479248047
# val 2.1162495613098145